# 07 — RL Budget Allocation Experiment

A 2-arm bandit decides per query whether to call the LLM or use embedding-only, maximising MRR under a fixed API call budget.

**Requires API key.**

In [ ]:
import sys; sys.path.insert(0, '..')
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.config import Settings
from src.data.fb15k237 import FB15k237Dataset
from src.models.embedding_baseline import RotatEBaseline
from src.models.llm_client import LLMClient
from src.models.reranker import LLMReranker
from src.rl.budget_agent import BudgetAgent
from src.rl.budget_experiment import BudgetExperiment
from src.rl.features import extract_features
from src.eval.candidates import generate_candidates
from src.eval.metrics import compute_metrics
from src.wikidata.sparql import WikidataGrounder
from src.wikidata.cache import SPARQLCache
from src.utils.cost_tracker import CostTracker
from src.utils.reproducibility import set_seed

settings = Settings()
set_seed(settings.random_seed)
assert settings.ai_gateway_api_key, 'Set AI_GATEWAY_API_KEY in .env!'

## 1. Load Components

In [ ]:
dataset = FB15k237Dataset(data_dir=settings.data_dir)
dataset.load()
embed_model = RotatEBaseline(cache_dir=settings.cache_dir)
embed_model.load()
cache    = SPARQLCache(cache_dir=settings.cache_dir)
grounder = WikidataGrounder(
    sparql_url=settings.wikidata_sparql_url,
    user_agent=settings.wikidata_user_agent,
    cache=cache,
)
client = LLMClient(settings=settings)
print('Components ready.')

## 2. Fixed-Budget Run

In [ ]:
BUDGET   = 10
random.seed(settings.random_seed)
queries  = random.sample(dataset.test, settings.sample_test_queries)
experiment = BudgetExperiment(
    embed_model=embed_model, client=client,
    grounder=grounder, dataset=dataset, settings=settings,
)
results = experiment.run(queries=queries, budget=BUDGET, template='minimal')
print(f'Budget used     : {results["budget_used"]}/{BUDGET}')
print(f'LLM reranked    : {results["llm_count"]}')
print(f'Embedding only  : {results["embed_count"]}')
print(f'MRR (RL policy) : {results["mrr"]:.4f}')
print(f'MRR (embed only): {results["embed_only_mrr"]:.4f}')

## 3. Budget Sweep: MRR vs Budget

In [ ]:
N = settings.sample_test_queries
budgets=[0,2,5,10,N]; mrrs=[]
for b in budgets:
    r=experiment.run(queries=queries,budget=b,template='minimal')
    mrrs.append(r['mrr'])
    print(f'  budget={b:3d}  MRR={r["mrr"]:.4f}')
fig,ax=plt.subplots(figsize=(8,4))
ax.plot(budgets,mrrs,marker='o',color='steelblue',linewidth=2)
ax.axhline(mrrs[0], color='gray', linestyle='--',label='Embed only')
ax.axhline(mrrs[-1],color='coral',linestyle='--',label='All LLM')
ax.set(title='MRR vs LLM Budget',xlabel='# queries reranked by LLM',ylabel='MRR')
ax.legend(); plt.tight_layout(); plt.show()

## 4. Which Queries Benefit Most?

In [ ]:
full = experiment.run(queries=queries,budget=N,template='minimal',return_per_query=True)
per_q = full.get('per_query',[])
if per_q:
    df=pd.DataFrame(per_q)
    df['rr_gain']=df['llm_rr']-df['embed_rr']
    display(df.sort_values('rr_gain',ascending=False).head(10)
              [['head','relation','embed_rank','llm_rank','rr_gain']])
else:
    print('Per-query breakdown not available.')

## 5. Agent Decision Boundary

In [ ]:
agent = BudgetAgent(total_budget=BUDGET)
diffs = np.linspace(0,1,50)
decs  = [agent.decide(query_difficulty=d) for d in diffs]
fig,ax=plt.subplots(figsize=(8,3))
ax.scatter(diffs,[int(d) for d in decs],
           c=['steelblue' if d else 'coral' for d in decs],s=60)
ax.set(title='Budget Agent: LLM Decision vs Difficulty',
       xlabel='Query difficulty',ylabel='Decision',
       yticks=[0,1],yticklabels=['Embed','LLM'])
plt.tight_layout(); plt.show()